In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append("..")

from runner import GridRunner, ExperimentSpec
from analysis import (
    discover_logs,
    load_many_logs,
    filter_runs,
    build_db_setup_matrix,
)
from evaluation import (
    relational_accuracy,
    symbiosis_score,
    quadratic_mean,
    exponential_decay,
)

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
spider_path = "../data/spider"
outputs_log_dir = "../outputs/logs_clean/enriched" # semantic grounding experiments
#outputs_log_dir = "../outputs/logs_clean/more_models" # candidate generation experiments

grid = GridRunner(
    spider_path=spider_path,
    outputs_log_dir=outputs_log_dir,
)

In [ ]:
# All the names of all the experiments (please just IGNORE the word "compact" in the names")
# Run this cell if you want to launch the experiment (see below)

specs = [
    
    # Baseline

    # ExperimentSpec(
    #     setup_name="baseline_4o_compact_one",
    #     agent_type="baseline",
    #     model_name="gpt-4o",
    #     temperature=0.7,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=1,
    #     max_workers_internal=1,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_4o_compact",
    #     agent_type="baseline",
    #     model_name="gpt-4o",
    #     temperature=0.7,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=5,
    #     max_workers_internal=5,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_4o_compact_hot",
    #     agent_type="baseline",
    #     model_name="gpt-4o",
    #     temperature=1.0,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=10,
    #     max_workers_internal=10,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_4o_compact_warm_heavy",
    #     agent_type="baseline",
    #     model_name="gpt-4o",
    #     temperature=0.7,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=20,
    #     max_workers_internal=20,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_compact_one",
    #     agent_type="baseline",
    #     model_name="gpt-4.1",
    #     temperature=0.0,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=1,
    #     max_workers_internal=1,
    # ),
     # ExperimentSpec(
    #     setup_name="baseline_compact",
    #     agent_type="baseline",
    #     model_name="gpt-4.1",
    #     temperature=0.7,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=5,
    #     max_workers_internal=5,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_compact_hot",
    #     agent_type="baseline",
    #     model_name="gpt-4.1",
    #     temperature=1.0,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=10,
    #     max_workers_internal=10,
    # ),
     # ExperimentSpec(
    #     setup_name="baseline_mini_compact_one",
    #     agent_type="baseline",
    #     model_name="gpt-4.1-mini",
    #     temperature=0.0,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=1,
    #     max_workers_internal=1,
    # ),
     # ExperimentSpec(
    #     setup_name="baseline_mini_compact",
    #     agent_type="baseline",
    #     model_name="gpt-4.1-mini",
    #     temperature=0.7,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=5,
    #     max_workers_internal=5,
    # ),
    # ExperimentSpec(
    #     setup_name="baseline_mini_compact_hot",
    #     agent_type="baseline",
    #     model_name="gpt-4.1-mini",
    #     temperature=1.0,
    #     memory_window=50,
    #     eval_limit=45,
    #     n_samples=10,
    #     max_workers_internal=10,
    # ),
]

In [ ]:
# Spider Databases used for the following experiments (details in the paper)

db_ids = [
    "concert_singer",
     "museum_visit",
     "student_transcripts_tracking",
]

In [ ]:
#Run this cell if you want to launch the experiment (An OpenAI key is needed)

# result = grid.run_grid(
#     db_ids=db_ids,
#     specs=specs,
#     max_workers=2,
#     resume=False,
# )

# result

In [ ]:
logs = discover_logs(outputs_log_dir, pattern="*.jsonl")
len(logs)

In [ ]:
multi_df = load_many_logs(logs)
print("rows:", len(multi_df))
multi_df.head()

In [ ]:
db_ids = [
 'student_transcripts_tracking',
 'concert_singer',
 'museum_visit',
]

In [ ]:
# You can choose the experiment (Run the cell to see the whole table)

df = filter_runs(
    multi_df,
    experiment_names=[
        #baseline
        
        # "baseline_4o_compact_one",
        # "baseline_4o_compact",
        # "baseline_4o_compact_hot",
        # "baseline_4o_compact_warm_heavy",
        # "baseline_compact_one",
        # "baseline_compact",
        # "baseline_compact_hot",
        # "baseline_mini_compact_one",
        # "baseline_mini_compact",
        # "baseline_mini_compact_hot",
    ],
    db_ids=db_ids,
)

print("filtered rows:", len(df))
df.head()

In [ ]:
def add_advanced_metrics(
    df: pd.DataFrame,
    *,
    alpha: float = 2.0,
    beta: float = 1.0,
    latency_cap: float = 5.0,
    token_cap: float = 10000.0,
    mu: float = 2.0,
):
    out = df.copy()

    rel_scores = []
    info_scores = []
    sym_scores = []

    for _, row in out.iterrows():
        pred = row.get("predicted_result")
        gold = row.get("gold_result")

        rel = relational_accuracy(
            predicted_result=pred,
            gold_result=gold,
            alpha=alpha,
            beta=beta,
        )

        info = relational_accuracy(
            predicted_result=pred,
            gold_result=gold,
            alpha=alpha,
            beta=0.0,
        )

        total_tokens = (row.get("tokens_input", 0) or 0) + (row.get("tokens_output", 0) or 0)

        sym = symbiosis_score(
            accuracy=info,
            resources={
                "latency": row.get("latency_sec"),
                "tokens": total_tokens,
            },
            caps={
                "latency": latency_cap,
                "tokens": token_cap,
            },
            weights={
                "latency": 1.0,
                "tokens": 1.0,
            },
            resource_agg_fn=quadratic_mean,
            decay_fn=exponential_decay,
            mu=mu,
        )

        rel_scores.append(rel)
        info_scores.append(info)
        sym_scores.append(sym["symbiosis"])

    out["relational_accuracy"] = rel_scores
    out["informational_relational_accuracy"] = info_scores
    out["symbiosis_score"] = sym_scores
    out["total_tokens"] = (
        pd.to_numeric(out["tokens_input"], errors="coerce").fillna(0.0)
        + pd.to_numeric(out["tokens_output"], errors="coerce").fillna(0.0)
    )

    return out

In [ ]:
df = add_advanced_metrics(df)
df.head()

In [ ]:
setup_summary = (
    df.groupby("experiment_name", dropna=False)
    .agg(
        num_turns=("turn_id", "count"),
        execution_ratio=("execution_match", lambda s: s.fillna(False).astype(float).mean()),
        relational_accuracy_mean=("relational_accuracy", "mean"),
        informational_relational_accuracy_mean=("informational_relational_accuracy", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
        symbiosis_score_mean=("symbiosis_score", "mean"),
    )
    .reset_index()
    #.sort_values("execution_ratio", ascending=False)
    .reset_index(drop=True)
)

setup_summary

In [ ]:
import pandas as pd
from IPython.display import display

def result_to_dataframe(result: dict | None) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()

    columns = result.get("columns", [])
    rows = result.get("rows", [])

    if not rows:
        return pd.DataFrame(columns=columns)

    # rows as dict
    if isinstance(rows[0], dict):
        df = pd.DataFrame(rows)
        if columns:
            ordered_cols = [c for c in columns if c in df.columns] + [c for c in df.columns if c not in columns]
            df = df[ordered_cols]
        return df

    # rows with lists/tuples
    return pd.DataFrame(rows, columns=columns if columns else None)


def inspect_turn_with_tables(df: pd.DataFrame, row_idx: int):
    row = df.iloc[row_idx]

    print("DB:", row["db_id"])
    print("TURN:", row["turn_id"])

    print("\nQUESTION:\n", row["question"])
    print("\nPREDICTED SQL:\n", row["predicted_sql"])
    print("\nGOLD SQL:\n", row["gold_sql"])

    print("\nEXECUTION:", row["execution_match"])
    print("RELATIONAL:", row["relational_accuracy"]) # <-- Choose the relational metric (by default A(2,1))
    print("TOOL CALLS:", row.get("num_tool_calls"))
    print("LATENCY:", row.get("latency_sec"))
    print("TOTAL TOKENS:", row.get("total_tokens"))

    pred_df = result_to_dataframe(row.get("predicted_result"))
    gold_df = result_to_dataframe(row.get("gold_result"))

    print("\n=== PREDICTED RESULT ===")
    print(f"rows: {len(pred_df)}, cols: {len(pred_df.columns)}")
    display(pred_df)

    print("\n=== GOLD RESULT ===")
    print(f"rows: {len(gold_df)}, cols: {len(gold_df.columns)}")
    display(gold_df)

    # simple set comparison, useful for intuition
    pred_tuples = set(map(tuple, pred_df.astype(str).itertuples(index=False, name=None))) if not pred_df.empty else set()
    gold_tuples = set(map(tuple, gold_df.astype(str).itertuples(index=False, name=None))) if not gold_df.empty else set()

    only_pred = pred_tuples - gold_tuples
    only_gold = gold_tuples - pred_tuples

    print("\n=== SET-LEVEL DIFFERENCE (stringified rows) ===")
    print("Rows only in predicted:", len(only_pred))
    for x in list(only_pred)[:10]:
        print("  PRED ONLY:", x)

    print("Rows only in gold:", len(only_gold))
    for x in list(only_gold)[:10]:
        print("  GOLD ONLY:", x)

In [ ]:
db_ids = [
 # 'student_transcripts_tracking',
 # 'concert_singer',
 # 'museum_visit',
]

In [ ]:
# Here you can choose which experiment you want to analyze in more detail

lot_cases = df[
    (df["experiment_name"] == "baseline_mini_compact_one") & (df["db_id"] == "concert_singer") &
    (df["execution_match"] == True) & (df["relational_accuracy"] >= 0.0 ) & (df["informational_relational_accuracy"] <= 1.0 )
].reset_index(drop=True)

for i in range(len(lot_cases)):
    inspect_turn_with_tables(lot_cases, row_idx=i)

In [ ]:
len(lot_cases)